# YOLO11s Baseline — RDD2022 India (4-class, D40 focus)

Trains the **baseline detector** on the frozen evaluation spine (Component 2B) on a Colab T4.
Run the cells top to bottom. Cell 3 is the actual (long) training run.

> **DELIBERATE BASELINE.** This run uses **default Ultralytics augmentation** and
> **NO custom D40 oversampling or class-imbalance handling**. Imbalance mitigation is a
> later *named ablation*, measured against this baseline — it is intentionally NOT baked in
> here. Do not add oversampling/class weights to this notebook.


## Cell 1 — clone/pull the repo and install dependencies

In [ ]:
# EDIT REPO_URL to your private repo. A GitHub token is needed for a private clone
# (Colab: use a fine-grained PAT, e.g. https://<TOKEN>@github.com/<you>/Pothole-Detection-System.git).
import os
REPO_URL = "https://github.com/<YOUR_USER>/Pothole-Detection-System.git"  # <-- EDIT
REPO_DIR = "/content/Pothole-Detection-System"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull
# Install ONLY ultralytics + sahi and keep Colab's own GPU-matched torch.
# (Do NOT pip install -r requirements.txt here: it pins torch==2.14.0, which can pull a
#  CUDA build mismatched to Colab's driver and silently fall back to CPU.)
!pip install -q ultralytics==8.4.160 sahi==0.12.6
%cd {REPO_DIR}

# Fail fast if there is no GPU (a CPU run of 100 epochs would be unusable).
import torch
assert torch.cuda.is_available(), "No CUDA GPU. Set Runtime > Change runtime type > GPU (T4) and re-run."
print("torch", torch.__version__, "| CUDA GPU:", torch.cuda.get_device_name(0))


## Cell 2 — Colab setup (Drive mount, data copy+unzip, yaml gen, font, checkpoints)

Edit the CONFIG block at the top of `notebooks/colab_setup.py` if your Drive/repo paths differ.

In [ ]:
import sys
sys.path.insert(0, "/content/Pothole-Detection-System/notebooks")
import colab_setup
cfg = colab_setup.setup()
cfg


## Cell 3 — YOLO11s baseline training

Parameters are the labeled block below. `resume=False` starts fresh; if the run is
interrupted, use the **Resume** cell instead of re-running this one.

In [ ]:
from ultralytics import YOLO

# ---------------- BASELINE PARAMS (frozen for the baseline run) ----------------
MODEL   = "yolo11s.pt"      # primary detector (project.md 21.1)
DATA    = cfg["data_yaml"]  # generated Colab-local yaml -> frozen split lists
IMGSZ   = 640
EPOCHS  = 100
BATCH   = 16
DEVICE  = 0                 # Colab GPU
PROJECT = cfg["project"]    # Drive dir -> checkpoints persist + resume
NAME    = "baseline"
# -------------------------------------------------------------------------------

model = YOLO(MODEL)
results = model.train(
    data=DATA, imgsz=IMGSZ, epochs=EPOCHS, batch=BATCH,
    device=DEVICE, project=PROJECT, name=NAME,
    resume=False,   # default augmentation; NO custom D40 oversampling (deliberate baseline)
)


## Resume cell — continue an interrupted run from `last.pt`

In [ ]:
from ultralytics import YOLO
last_ckpt = f"{cfg['project']}/baseline/weights/last.pt"
model = YOLO(last_ckpt)
model.train(resume=True)


## Cell 4 — evaluate on the FROZEN TEST split

Reports mAP@50, mAP@50:95, and per-class AP (especially **D40**). This is the primary
mAP spine; the size-stratified small-recall CI uses the 5-fold CV assignment separately.

In [ ]:
from ultralytics import YOLO
best_ckpt = f"{cfg['project']}/baseline/weights/best.pt"
model = YOLO(best_ckpt)
metrics = model.val(data=cfg["data_yaml"], split="test", imgsz=640, device=0)

print("mAP@50    :", round(float(metrics.box.map50), 4))
print("mAP@50-95 :", round(float(metrics.box.map), 4))
print("per-class AP (index -> name : AP50 / AP50-95):")
for i, c in enumerate(metrics.box.ap_class_index):
    print(f"  {c} {metrics.names[int(c)]:>4} : {metrics.box.ap50[i]:.4f} / {metrics.box.ap[i]:.4f}")
